In [1]:
import pandas as pd
import openpyxl

In [15]:
df = pd.read_excel("/home/callum/sno_data/raw/shu.xlsx", header=0, sheet_name = "Original MS data")


In [ ]:
# df = df.rename(columns = {"Position": "position", "Amino acid": "aa"})
# print(df.columns)



Index(['Protein accession', 'position', 'aa', 'Protein description',
       'Localization probability', 'PEP', 'Score', 'Modified sequence',
       'Charge', 'Mass error [ppm]', 'MS/MS Count', 'XN_1 intensity',
       'XN_2 intensity', 'KEGG Gene', 'Unnamed: 14'],
      dtype='str')


In [18]:
import time
import pandas as pd
import requests

API = "https://rest.uniprot.org/idmapping"

def run_job(ids, from_db, to_db):
    r = requests.post(f"{API}/run", data={"from": from_db, "to": to_db, "ids": ",".join(ids)})
    r.raise_for_status()
    job = r.json()["jobId"]
    for _ in range(60):
        s = requests.get(f"{API}/status/{job}", allow_redirects=False)
        if s.status_code == 303 or "FINISHED" in s.text:
            break
        if "FAILED" in s.text or "ERROR" in s.text:
            raise RuntimeError(s.text)
        time.sleep(2)
    results, suggested, url = [], [], f"{API}/results/{job}?size=500"
    while url:                                   # follow pagination
        r = requests.get(url); r.raise_for_status()
        j = r.json()
        results += j.get("results", [])
        suggested += j.get("suggestedIds", [])
        url = r.links.get("next", {}).get("url")
    return results, suggested

ids = df[ID].dropna().drop_duplicates().tolist()   # your GenBank IDs, e.g. AID38452.1

# step 1: GenBank protein ID -> UniProtKB directly, else UniParc
direct_res, to_upi = run_job(ids, "EMBL-GenBank-DDBJ_CDS", "UniProtKB")
upi_map = {x["from"]: x["to"] for x in to_upi}              # accession -> UPI

# step 2: UniParc -> UniProtKB (identical sequence in another entry)
upis = sorted(set(upi_map.values()))
via_res, _ = run_job(upis, "UniParc", "UniProtKB") if upis else ([], [])
import io

def acc_of(e):
    return e["primaryAccession"] if isinstance(e, dict) else e

pairs = [(x["from"], acc_of(x["to"]), "direct") for x in direct_res]
for x in via_res:
    for acc, upi in upi_map.items():
        if upi == x["from"]:
            pairs.append((acc, acc_of(x["to"]), "via_uniparc"))

cand = pd.DataFrame(pairs, columns=["src", "uniprot_id", "route"])

def fetch_details(accs, batch=100):
    frames = []
    for i in range(0, len(accs), batch):
        r = requests.get("https://rest.uniprot.org/uniprotkb/accessions",
                         params={"accessions": ",".join(accs[i:i+batch]), "format": "tsv",
                                 "fields": "accession,reviewed,length,organism_name"}, timeout=60)
        r.raise_for_status()
        frames.append(pd.read_csv(io.StringIO(r.text), sep="\t"))
    return pd.concat(frames, ignore_index=True)

if len(cand):
    det = fetch_details(cand["uniprot_id"].unique().tolist())
    cand = cand.merge(det.rename(columns={"Entry": "uniprot_id"}), on="uniprot_id", how="left")
    cand["is_reviewed"] = cand["Reviewed"].eq("reviewed")
    cand["direct"] = cand["route"].eq("direct")
    n_cand = cand["src"].value_counts()
    best = (cand.sort_values(["src", "direct", "is_reviewed", "Length"],
                             ascending=[True, False, False, False])
                .drop_duplicates("src"))
    best["n_candidates"] = best["src"].map(n_cand)
    best = best.rename(columns={"src": ID, "Reviewed": "reviewed", "Organism": "organism"})
    best = best[[ID, "uniprot_id", "reviewed", "organism", "route", "n_candidates"]]
else:
    best = pd.DataFrame(columns=[ID, "uniprot_id", "reviewed", "organism", "route", "n_candidates"])

out = pd.DataFrame({ID: ids})
out["uniparc_id"] = out[ID].map(upi_map)
out = out.merge(best, on=ID, how="left")
out["route"] = out["route"].fillna("uniparc_only")

print(out["route"].value_counts())
out

route
via_uniparc     245
uniparc_only     58
Name: count, dtype: int64


,Protein accession,uniparc_id,uniprot_id,reviewed,organism,route,n_candidates
0,AID38461.1,UPI000005499D,NaN,NaN,NaN,uniparc_only,NaN
1,AID39171.1,UPI0002506E8A,NaN,NaN,NaN,uniparc_only,NaN
2,AID38447.1,UPI0000ED915B,Q6GKU0,reviewed,Staphylococcus aureus (strain MRSA252),via_uniparc,1.0
3,AID38450.1,UPI00000D9B7B,A0A7U7JS23,unreviewed,Staphylococcus argenteus,via_uniparc,1.0
4,AID38452.1,UPI00003B12E4,Q6GKT6,reviewed,Staphylococcus aureus (strain MRSA252),via_uniparc,1.0
...,...,...,...,...,...,...,...
298,AID41343.1,UPI00000CAC67,Q99R06,reviewed,Staphylococcus aureus (strain Mu50 / ATCC 700699),via_uniparc,7.0
299,AID41356.1,UPI00003B19D8,NaN,NaN,NaN,uniparc_only,NaN
300,AID41357.1,UPI00000D7800,A0A0H2XFN8,unreviewed,Staphylococcus aureus (strain USA300),via_uniparc,2.0
301,AID41403.1,UPI00019CD62B,NaN,NaN,NaN,uniparc_only,NaN


In [33]:
# filter based on uniprot ids

subset = out.dropna(subset=["uniprot_id"])
print (subset.head()
       )

  Protein accession     uniparc_id  uniprot_id    reviewed  \
0        AID38461.1  UPI000005499D      A8YYU1         NaN   
2        AID38447.1  UPI0000ED915B      Q6GKU0    reviewed   
3        AID38450.1  UPI00000D9B7B  A0A7U7JS23  unreviewed   
4        AID38452.1  UPI00003B12E4      Q6GKT6    reviewed   
5        AID38460.1  UPI0000132B0C      P65884    reviewed   

                                            organism        route  \
0                                                NaN   similarity   
2             Staphylococcus aureus (strain MRSA252)  via_uniparc   
3                           Staphylococcus argenteus  via_uniparc   
4             Staphylococcus aureus (strain MRSA252)  via_uniparc   
5  Staphylococcus aureus (strain Mu50 / ATCC 700699)  via_uniparc   

   n_candidates uniprot_sim  pident  
0           NaN      A8YYU1   100.0  
2           1.0         NaN     NaN  
3           1.0         NaN     NaN  
4           1.0         NaN     NaN  
5           3.0       

In [ ]:
# going to stop here- lots of different strains in this data and some are unknown - already lost many sites due to not being able to map to uniprot id so will think about these 

diamond v2.1.11.165 (C) Max Planck Society for the Advancement of Science, Benjamin Buchfink, University of Tuebingen
Documentation, support and updates available at http://www.diamondsearch.org
Please cite: http://dx.doi.org/10.1038/s41592-021-01101-x Nature Methods (2021)

#CPU threads: 32
Scoring parameters: (Matrix=BLOSUM62 Lambda=0.267 K=0.041 Penalties=11/1)
Database input file: work/Staphylococcus_aureus_ref.fasta
Opening the database file...  [0.001s]
Loading sequences...  [0.012s]
Masking sequences...  [0.009s]
Writing sequences...  [0.001s]
Hashing sequences...  [0s]
Loading sequences...  [0s]
Writing trailer...  [0s]
Closing the input file...  [0s]
Closing the database file...  [0s]

Database sequences  10269
  Database letters  3427968
     Database hash  cbc4ecd48b81e2e7b223ec2b492c3468
        Total time  0.026000s
diamond v2.1.11.165 (C) Max Planck Society for the Advancement of Science, Benjamin Buchfink, University of Tuebingen
Documentation, support and updates availa

route
via_uniparc     245
uniparc_only     31
similarity       27
Name: count, dtype: int64
235 233 False


In [11]:
print(r.status_code, r.text[:300])

NameError: name 'r' is not defined